In [1]:
import pandas as pd
import json

centerline_json = "./data/raw/centerlines/Jefferson_County_KY_Street_Centerlines.geojson"
with open(centerline_json) as data:
    json_data = json.load(data)


In [28]:
# extract properties for each road from data

centerline_properties = pd.DataFrame.from_dict([feature['properties'] for feature in json_data['features']])
centerline_properties = centerline_properties.set_index('OBJECTID')
centerline_properties.head()

,RWCOMPKEY,RWCOMPKEYPARENT,LOW_ADDRESS,FROM_ADDRESS,TO_ADDRESS,LFROM,LTO,RFROM,RTO,SIFID,...,ALIAS3_DIR,ALIAS3_STRNAME,ALIAS3_TYPE,ALIAS3_SIFID,ALIAS4_DIR,ALIAS4_STRNAME,ALIAS4_TYPE,ALIAS4_SIFID,GLOBALID,SHAPELEN
OBJECTID,,,,,,,,,,,,,,,,,,,,,
1,24074,0,6300,6300,6399,6301,6399,6300,6398,8665,...,,,,0,,,,0,{496EE316-2749-408F-AD2D-16DD49C99017},325.140306
2,20311,0,1600,1600,1619,1601,1619,1600,1618,5926,...,,,,0,,,,0,{AA2CAA18-473D-416D-8973-C74538F2BB31},447.052294
3,21109,0,1700,1700,1705,1701,1705,1700,1704,473,...,,,,0,,,,0,{A9B57033-3F9C-4CA7-8ED3-0F17D5DF6C14},552.489593
4,20926,0,505,505,519,505,519,506,518,2442,...,,,,0,,,,0,{F40EEA69-C8E4-4A2B-983D-670A93C87BFE},193.929089
5,19036,0,730,730,799,731,799,730,798,4573,...,,,,0,,,,0,{D0515DFE-8CF3-450F-9470-A1C35303ACA7},125.862060


In [29]:
INDEX = 'OBJECTID'

SIF_INFO = ['SIFID', 'SIFCODE', 'SIFIDLOW', 'LOCROSSSIF', 'SIFIDHI', 'HICROSSSIF']

# name info helpful for presentation but no so much for searching road network
# probably keep on of these like ROADNAME in working data for sanity checks but most of
# this can be looked up on an as needed basis. 
# Move into a separate linked database. 
NAME_INFO = ['ROADNAME']

NAME_INFO_TO_DROP = ['AVDIR', 'AVPRETYPE', 'AVSTNAME', 'AVSTTYPE', 'AVSUFDIR', 
                     'DIR', 'STRNAME', 'TYPE', 'ROUTE_LBL']

# can always look this up other ways (probably)
CROSS_STREET_NAMES = ['LOCROSSPR', 'LOCROSSNA', 'LOCROSSSU','HICROSSPR', 'HICROSSNA', 'HICROSSSU']


# Use this to filter out stuff like interstates
ROAD_CLASS = ['CORE_CLASS']
ROAD_CLASS_REDUNDANT = ['CLASS_ABRV', 'STCL_CLASS']


# seems like it should be helpful but I can't see how exactly.
GEOMETRY_INFO = ['LENGTH', 'SHAPELEN']


# find out the structure here?
MIDAS = ['COMPKEY', 'COMPTYPE', 'UNITID', 'UNITID2', 'RWCOMPKEY', 'RWCOMPKEYPARENT']

# what do these mean?
ALIASES = ['ALIAS1_DIR', 'ALIAS1_STRNAME', 'ALIAS1_TYPE', 'ALIAS1_SIFID',
           'ALIAS2_DIR', 'ALIAS2_STRNAME', 'ALIAS2_TYPE', 'ALIAS2_SIFID',
           'ALIAS3_DIR', 'ALIAS3_STRNAME', 'ALIAS3_TYPE', 'ALIAS3_SIFID',
           'ALIAS4_DIR', 'ALIAS4_STRNAME', 'ALIAS4_TYPE', 'ALIAS4_SIFID']

# not helpful to me
ADDRESS_INFO = ['LOW_ADDRESS','FROM_ADDRESS', 'TO_ADDRESS', 
                'LFROM', 'LTO', 'RFROM', 'RTO', 'ZIP_LEFT', 'ZIP_RIGHT']
# no need to look up house addresses

SPEED_INFO = ['FTIMP', 'TFIMP', 'SPEED'] 
# don't care about speed

CITY_USE = ['MUNI_NAME', 'COUNDIST', 'MAIN_DIST', 'OWNER_NAME', 'OWNER_ABRV',
             'MAIN_RESP', 'MAIN_ABRV', 'ONE_WAY', 'MAPNO', 'GLOBALID']
# dont care about city maintenance districts/owners or what have you
# don't care about one ways
# don't know what globalid is but it seems to to with LOJIC internally. Same with MAPNO

# simply not useful to me:
DATABASE_META = ['ENTERED_BY', 'ENTERED_ON', 'UPDATED_BY', 'UPDATED_ON']


In [36]:

exclusion_lists = [ADDRESS_INFO, SPEED_INFO, CITY_USE, DATABASE_META, GEOMETRY_INFO,
            CROSS_STREET_NAMES, NAME_INFO_TO_DROP, ROAD_CLASS_REDUNDANT]
# minimal centerline data: may bring some of these back at some point:
exclusion_lists += [ALIASES, MIDAS]

exclusions = list()
for sublist in exclusion_lists:
    exclusions.extend(sublist)
columns = centerline_properties.columns
columns = columns.drop(exclusions)
columns


Index(['SIFID', 'SIFCODE', 'ROADNAME', 'SIFIDLOW', 'LOCROSSSIF', 'SIFIDHI',
       'HICROSSSIF', 'CORE_CLASS'],
      dtype='object')

In [37]:
# filter data to just what I'll need.

centerlines_network = centerline_properties[['ROADNAME', 'CORE_CLASS', 'SIFID', 'SIFCODE', 
 'SIFIDLOW', 'LOCROSSSIF', 'SIFIDHI', 'HICROSSSIF']]
centerlines_network.head()


,ROADNAME,CORE_CLASS,SIFID,SIFCODE,SIFIDLOW,LOCROSSSIF,SIFIDHI,HICROSSSIF
OBJECTID,,,,,,,,
1,SERENITY CT,LOCAL,8665,9887,1550,1557,8594,9811
2,S 28TH ST,LOCAL,5926,6570,2854,2934,2470,2499
3,BEECH ST,LOCAL,473,0458,6487,7212,10596,D596
4,GARDEN DR,PRIMARY COLLECTOR,2442,2470,13394,5391,4702,5128
5,PARKWAY DR,LOCAL,4573,4974,4162,4476,8594,9811


In [32]:
# collect unused data in its own table. Why not?

# extra columns:
['LOW_ADDRESS', 'FROM_ADDRESS', 'TO_ADDRESS', 'LFROM', 'LTO', 'RFROM',
        'RTO', 'ZIP_LEFT', 'ZIP_RIGHT', 'FTIMP', 'TFIMP', 'SPEED', 'MUNI_NAME',
        'COUNDIST', 'MAIN_DIST', 'OWNER_NAME', 'OWNER_ABRV', 'MAIN_RESP',
        'MAIN_ABRV', 'ONE_WAY', 'MAPNO', 'GLOBALID', 'ENTERED_BY', 'ENTERED_ON',
        'UPDATED_BY', 'UPDATED_ON', 'LENGTH', 'SHAPELEN', 'LOCROSSPR',
        'LOCROSSNA', 'LOCROSSSU', 'HICROSSPR', 'HICROSSNA', 'HICROSSSU',
        'AVDIR', 'AVPRETYPE', 'AVSTNAME', 'AVSTTYPE', 'AVSUFDIR', 'DIR',
        'STRNAME', 'TYPE', 'ROUTE_LBL', 'CLASS_ABRV', 'STCL_CLASS',
        'ALIAS1_DIR', 'ALIAS1_STRNAME', 'ALIAS1_TYPE', 'ALIAS1_SIFID',
        'ALIAS2_DIR', 'ALIAS2_STRNAME', 'ALIAS2_TYPE', 'ALIAS2_SIFID',
        'ALIAS3_DIR', 'ALIAS3_STRNAME', 'ALIAS3_TYPE', 'ALIAS3_SIFID',
        'ALIAS4_DIR', 'ALIAS4_STRNAME', 'ALIAS4_TYPE', 'ALIAS4_SIFID',
        'COMPKEY', 'COMPTYPE', 'UNITID', 'UNITID2', 'RWCOMPKEY',
        'RWCOMPKEYPARENT']
#


centerlines_extra_data = centerline_properties[exclusions]
centerlines_extra_data.head()


,LOW_ADDRESS,FROM_ADDRESS,TO_ADDRESS,LFROM,LTO,RFROM,RTO,ZIP_LEFT,ZIP_RIGHT,FTIMP,...,ALIAS4_DIR,ALIAS4_STRNAME,ALIAS4_TYPE,ALIAS4_SIFID,COMPKEY,COMPTYPE,UNITID,UNITID2,RWCOMPKEY,RWCOMPKEYPARENT
OBJECTID,,,,,,,,,,,,,,,,,,,,,
1,6300,6300,6399,6301,6399,6300,6398,40219,40219,0.246000,...,,,,0,270349312,68,9887,0002,24074,0
2,1600,1600,1619,1601,1619,1600,1618,40211,40211,0.339000,...,,,,0,270344558,68,6570,0021,20311,0
3,1700,1700,1705,1701,1705,1700,1704,40211,40211,0.415000,...,,,,0,270345921,68,0458,0011,21109,0
4,505,505,519,505,519,506,518,40206,40206,0.073458,...,,,,0,270342861,68,2470,0006,20926,0
5,730,730,799,731,799,730,798,40217,40217,0.095000,...,,,,0,270343420,68,4974,0012,19036,0


In [56]:
# extract geometries from JSON

def get_geometry(data):
    for feature in data['features']:
        properties = feature['properties']
        objectid = properties['OBJECTID']
        geo = feature['geometry']['coordinates']
        yield(objectid, geo)

centerlines_geo = pd.DataFrame.from_records(get_geometry(json_data), columns=["OBJECTID", "GEOMETRY"])
centerlines_geo = centerlines_geo.set_index("OBJECTID")

centerlines_geo.head()


,GEOMETRY
OBJECTID,
1,"[[-85.6809503218278, 38.158867088749105], [-85..."
2,"[[-85.80120122370631, 38.23056379303306], [-85..."
3,"[[-85.80501498815028, 38.228933021550915], [-8..."
4,"[[-85.68020545343364, 38.24806713661984], [-85..."
5,"[[-85.74203979766622, 38.211814770925905], [-8..."


In [57]:
# combine selected properties with the geometry for each segment into one table.
centerlines_data = pd.concat((centerlines_network, centerlines_geo), axis=1)
centerlines_data.head()

,ROADNAME,CORE_CLASS,SIFID,SIFCODE,SIFIDLOW,LOCROSSSIF,SIFIDHI,HICROSSSIF,GEOMETRY
OBJECTID,,,,,,,,,
1,SERENITY CT,LOCAL,8665,9887,1550,1557,8594,9811,"[[-85.6809503218278, 38.158867088749105], [-85..."
2,S 28TH ST,LOCAL,5926,6570,2854,2934,2470,2499,"[[-85.80120122370631, 38.23056379303306], [-85..."
3,BEECH ST,LOCAL,473,0458,6487,7212,10596,D596,"[[-85.80501498815028, 38.228933021550915], [-8..."
4,GARDEN DR,PRIMARY COLLECTOR,2442,2470,13394,5391,4702,5128,"[[-85.68020545343364, 38.24806713661984], [-85..."
5,PARKWAY DR,LOCAL,4573,4974,4162,4476,8594,9811,"[[-85.74203979766622, 38.211814770925905], [-8..."


In [60]:
# write cleaner files for future use
save_location = "/Users/bencampbell/code/county_coverage/data/cleaner/{filename}.json"

#centerlines_network.to_json(save_location.format(filename="centerlines_network"))

#centerlines_extra_data.to_json(save_location.format(filename="centerlines_extra_data"))

#centerlines_geo.to_json(save_location.format(filename="centerlines_geo"))

centerlines_data.to_json(save_location.format(filename="centerlines_data"))



In [35]:
# read test

#pd.read_json("/Users/bencampbell/code/county_coverage/data/cleaner/centerlines_network.json").head()
# works no problem!

# OBJECTID column name is not preserved
# not a big deal since the index is consistent between files